[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ZeruiW/frontier-ai-courses/blob/main/C28_Frontier_Diffusion_Course/04_guidance/04_guidance.ipynb)

# 04 · Classifier-Free Guidance（用 numpy 玩具复现）

目标：用纯 numpy 把 **CFG**从零跑通：① 构造带条件的 2D 玩具与条件/无条件去噪器；② 实现 CFG 外推 `ε̃=ε_∅+w(ε_c−ε_∅)`；③ **扫描引导强度 w**，亲眼量化「保真↑、多样↓」的权衡；④ 负引导。

路线：条件玩具数据 → 条件/无条件去噪器 → CFG 外推(验 w=0/w=1 边界) → 强度扫描(保真-多样权衡) → 负引导 → 动态 CFG → ✏️ 练习 → 📖 答案 → 🧪 真实引导强度胶囊。

> **统一约定**（与讲解一致）：`ε̃ = ε_∅ + w·(ε_c − ε_∅)`。w=0→无条件，w=1→普通条件，w>1→放大条件。
> 心智模型：**CFG = 问模型『随便画』和『按 prompt 画』，把两者之差放大 w 倍**。维度小到能画散点，但外推式与真实 SD 一行 CFG 代码逐字对应。

## 1 · 带条件的 2D 玩具：两个类别

CFG 需要「条件」。我们造最简单的条件玩具：**两个分得很开的高斯团**当两个类别 `c∈{0,1}`；它们的**混合**就是「无条件」分布（不指定类别时该长什么样）。

In [ ]:
import numpy as np
rng = np.random.default_rng(0)

# 两个类别中心：适度分离（留一点重叠，让 CFG 的「逐步收紧」可见）
CENTERS = np.array([[-1.0, 0.0],    # 类别 0
                    [ 1.0, 0.0]])   # 类别 1
CLASS_STD = 0.8

def sample_class(c, n, rng):
    '''从类别 c 采 n 个 2D 点。'''
    return CENTERS[c] + CLASS_STD * rng.standard_normal((n, 2))

def sample_unconditional(n, rng):
    '''无条件 = 两类等概率混合。'''
    cs = rng.integers(0, 2, size=n)
    return CENTERS[cs] + CLASS_STD * rng.standard_normal((n, 2)), cs

X0 = sample_class(0, 2000, rng)
X1 = sample_class(1, 2000, rng)
Xu, cu = sample_unconditional(4000, rng)
print('类别0 均值', np.round(X0.mean(0), 2), ' 类别1 均值', np.round(X1.mean(0), 2))
print('无条件 均值', np.round(Xu.mean(0), 2), '(≈两中心的平均 [0,0])')
assert np.allclose(X0.mean(0), CENTERS[0], atol=0.1)
assert np.allclose(X1.mean(0), CENTERS[1], atol=0.1)
assert np.allclose(Xu.mean(0), [0, 0], atol=0.15), '混合均值应≈两中心平均'
print('✅ 条件玩具就绪：c=0/1 两个分离类别，混合=无条件分布')

## 2 · 条件 / 无条件去噪器（闭式后验）

真实 SD 用一个网络（带条件丢弃训练）同时给出 `ε_c` 和 `ε_∅`。玩具里我们直接用**闭式后验均值**构造它们：

- 条件去噪器：已知类别 c，先验是 `N(CENTERS[c], σ²I)`，对 `x_t=√ᾱ·x0+√(1-ᾱ)·ε` 算后验 `E[x0|x_t,c]`，反解 `ε_c`。
- 无条件去噪器：先验是两类混合，用**后验加权**（responsibility）混合两个类别的预测得到 `ε_∅`。

In [ ]:
betas = np.linspace(1e-4, 0.02, 200)
abar = np.cumprod(1.0 - betas)

def q_sample(x0, t, eps, abar):
    a = abar[t]
    return np.sqrt(a) * x0 + np.sqrt(1 - a) * eps

def cond_x0_hat(x_t, t, c, abar, prior_std=CLASS_STD):
    '''高斯先验 N(mu_c, s²I) 下 E[x0|x_t]。'''
    a = abar[t]; s2 = prior_std**2; mu = CENTERS[c]
    # 后验均值（标准高斯-高斯共轭）
    var_post = 1.0 / (a / (1 - a) + 1.0 / s2)
    mean_post = var_post * (np.sqrt(a) / (1 - a) * x_t + mu / s2)
    return mean_post

def eps_cond(x_t, t, c, abar):
    '''条件 ε 预测：由 x_t 与 x0_hat 反解。'''
    a = abar[t]
    x0h = cond_x0_hat(x_t, t, c, abar)
    return (x_t - np.sqrt(a) * x0h) / np.sqrt(1 - a)

def eps_uncond(x_t, t, abar):
    '''无条件 ε：两类后验责任加权混合。'''
    a = abar[t]
    # 每类的边际似然 p(x_t|c) ∝ N(x_t; √a·mu_c, (a·s²+(1-a)) I)
    var_c = a * CLASS_STD**2 + (1 - a)
    logp = []
    for cc in [0, 1]:
        d2 = ((x_t - np.sqrt(a) * CENTERS[cc])**2).sum(-1)
        logp.append(-0.5 * d2 / var_c)
    logp = np.stack(logp, -1)                 # (n,2)
    w = np.exp(logp - logp.max(-1, keepdims=True))
    resp = w / w.sum(-1, keepdims=True)        # 责任 (n,2)
    eps0 = eps_cond(x_t, t, 0, abar)
    eps1 = eps_cond(x_t, t, 1, abar)
    return resp[:, [0]] * eps0 + resp[:, [1]] * eps1

# 自检：条件去噪器把含噪样本的均值拉回类别中心（用独立 rng + 大样本，结论稳健）
g_chk = np.random.default_rng(123)
t = 120                                   # 中等噪声，去噪效果明显
x0 = sample_class(1, 5000, g_chk)
eps = g_chk.standard_normal(x0.shape)
x_t = q_sample(x0, t, eps, abar)
x0h = cond_x0_hat(x_t, t, 1, abar)
d_xt  = np.linalg.norm(x_t.mean(0) - CENTERS[1])
d_x0h = np.linalg.norm(x0h.mean(0) - CENTERS[1])
print(f'含噪 x_t 均值到类别1 距离 {d_xt:.3f} -> 去噪后 x0_hat 距离 {d_x0h:.3f}')
assert d_x0h < d_xt, '条件去噪应把均值拉回类别中心'
print(f'类别1 去噪 x0_hat 均值 {np.round(x0h.mean(0),2)} (真中心 {CENTERS[1]})')
print('✅ 条件/无条件去噪器就绪（闭式后验，当作「完美训练好的网络」）')

## 3 · CFG 外推：`ε̃ = ε_∅ + w·(ε_c − ε_∅)`

核心一行。验证两个**精确边界等式**（这是 CFG 正确性的基石）：
- `w=0` → `ε̃ = ε_∅`（纯无条件）
- `w=1` → `ε̃ = ε_c`（普通条件，不放大）
- `w>1` → 外推到 `ε_c` 之外（沿 `ε_∅→ε_c` 方向继续走）。

In [ ]:
def cfg_predict(eps_c, eps_uncond_, w):
    '''统一约定：ε̃ = ε_∅ + w·(ε_c − ε_∅)。'''
    return eps_uncond_ + w * (eps_c - eps_uncond_)

t = 60
x_t = rng.standard_normal((200, 2))
ec = eps_cond(x_t, t, 1, abar)      # 目标类别 1
eu = eps_uncond(x_t, t, abar)

# 边界等式
assert np.allclose(cfg_predict(ec, eu, 0.0), eu), 'w=0 必须等于无条件'
assert np.allclose(cfg_predict(ec, eu, 1.0), ec), 'w=1 必须等于条件'
# w>1 是外推：ε̃ 在 ε_c 之外，且 (ε̃-ε_c) 与 (ε_c-ε_∅) 同向
et = cfg_predict(ec, eu, 4.0)
beyond = et - ec
direction = ec - eu
cos = (beyond * direction).sum() / (np.linalg.norm(beyond) * np.linalg.norm(direction) + 1e-12)
print(f'w=4 外推方向与 (ε_c-ε_∅) 的余弦 = {cos:.3f} (应≈1, 同向外推)')
assert cos > 0.99, 'w>1 应沿 ε_∅→ε_c 方向继续外推'
# 外推幅度随 w 线性增长
norms = [np.linalg.norm(cfg_predict(ec, eu, w) - ec) for w in [1, 2, 4, 8]]
print('|ε̃-ε_c| @ w=1,2,4,8:', np.round(norms, 3))
assert norms[0] < 1e-9 and norms[-1] > norms[1] > 1e-9
print('✅ CFG 外推正确：w=0→ε_∅、w=1→ε_c、w>1 沿条件方向外推（幅度∝w-1）')

## 4 · 带 CFG 的采样

把 CFG 预测喂进 DDPM 采样：每步算 `ε_c` 和 `ε_∅`，外推得 `ε̃`，用 `ε̃` 走一步。

先跑通：用较大的 `w` 朝类别 1 采样，验证生成样本确实聚到类别 1 的中心附近。

In [ ]:
alphas = 1.0 - betas

def sample_cfg(n, target_c, w, seed=1):
    g = np.random.default_rng(seed)
    x = g.standard_normal((n, 2))
    for t in range(len(betas) - 1, -1, -1):
        ec = eps_cond(x, t, target_c, abar)
        eu = eps_uncond(x, t, abar)
        eps_hat = cfg_predict(ec, eu, w)
        a_t, ab_t = alphas[t], abar[t]
        mean = (x - (1 - a_t) / np.sqrt(1 - ab_t) * eps_hat) / np.sqrt(a_t)
        x = mean + (np.sqrt(betas[t]) * g.standard_normal(x.shape) if t > 0 else 0.0)
    return x

gen1 = sample_cfg(1500, target_c=1, w=4.0)
print('朝类别1 (w=4) 生成均值:', np.round(gen1.mean(0), 2), ' 真中心', CENTERS[1])
# 应聚到类别 1 附近，远离类别 0
d_to_1 = np.linalg.norm(gen1.mean(0) - CENTERS[1])
d_to_0 = np.linalg.norm(gen1.mean(0) - CENTERS[0])
assert d_to_1 < d_to_0, '朝类别1 引导应更靠近类别1'
assert d_to_1 < 1.0, '应明显聚到目标类别附近'
print('✅ 带 CFG 采样跑通：朝目标类别生成，聚集到其中心附近')

## 5 · 强度扫描：保真 vs 多样的权衡（核心）

本模块的灵魂实验。对 `w∈{0,1,2,4,8}` 各采一大批样本（朝类别 1），测两个指标：
- **保真度**：落在目标类别一侧（`x>0`）的样本比例（越高越扣条件）。
- **多样性**：样本沿区分轴（x 轴）的标准差（越大越散；越小越聚到目标）。

预期：**w↑ → 保真↑、多样↓**，且都**单调**。这就是你在 UI 里调 CFG 滑块时的真实权衡。

In [ ]:
def fidelity_and_diversity(samples, target_c):
    '''保真=落在目标类别一侧的比例；多样=沿区分轴(x)的 std。'''
    if target_c == 1:
        on_target = (samples[:, 0] > 0).mean()      # 类别1 在 x>0
    else:
        on_target = (samples[:, 0] < 0).mean()
    diversity = samples[:, 0].std()                 # 沿区分轴的散布
    return on_target, diversity

print(f"{'w':>4} {'保真(在目标侧%)':>16} {'多样(std_x)':>13}")
fids, divs = [], []
for w in [0.0, 1.0, 2.0, 4.0, 8.0]:
    s = sample_cfg(3000, target_c=1, w=w, seed=2)
    fid, div = fidelity_and_diversity(s, 1)
    fids.append(fid); divs.append(div)
    print(f'{w:>4.0f} {fid*100:>15.1f}% {div:>13.3f}')

# 保真度：w=0(无条件)约半数在目标侧；随 w 单调上升到≈100%
assert fids[0] < 0.7, 'w=0 无条件，约半数在目标侧'
assert all(fids[i] <= fids[i+1] + 1e-9 for i in range(len(fids)-1)), '保真应随 w 单调不减'
assert fids[-1] > 0.95, '大 w 应几乎全部落在目标侧'
# 多样性：沿区分轴的散布随 w 单调下降（越来越聚到目标类）
assert all(divs[i] >= divs[i+1] - 1e-9 for i in range(len(divs)-1)), '多样性应随 w 单调不增'
assert divs[-1] < divs[0] * 0.6, 'w 大时明显比无条件更集中'
print('\n✅ 保真-多样权衡的数字证据：w↑ 保真↑ 多样↓（均单调）—— 这就是 CFG 滑块的本质')

## 6 · 负向提示：把基线换成「不想要的类别」

负 prompt：把外推基线从 `ε_∅` 换成「不想要内容」的预测 `ε_neg`：`ε̃ = ε_neg + w·(ε_c − ε_neg)`。

实验：正条件=类别1、负条件=类别0。对比「普通 CFG（基线=无条件）」与「负引导（基线=类别0）」，验证负引导把样本**更强地推离类别0**。

In [ ]:
def sample_cfg_negative(n, target_c, neg_c, w, seed=1):
    g = np.random.default_rng(seed)
    x = g.standard_normal((n, 2))
    for t in range(len(betas) - 1, -1, -1):
        ec = eps_cond(x, t, target_c, abar)
        eneg = eps_cond(x, t, neg_c, abar)          # 基线=负类别
        eps_hat = eneg + w * (ec - eneg)
        a_t, ab_t = alphas[t], abar[t]
        mean = (x - (1 - a_t) / np.sqrt(1 - ab_t) * eps_hat) / np.sqrt(a_t)
        x = mean + (np.sqrt(betas[t]) * g.standard_normal(x.shape) if t > 0 else 0.0)
    return x

w = 3.0
plain = sample_cfg(2000, target_c=1, w=w, seed=5)              # 基线=无条件
neg   = sample_cfg_negative(2000, target_c=1, neg_c=0, w=w, seed=5)  # 基线=类别0
# 负引导应让样本均值离类别0 更远（被推开）
d_plain_to_neg = np.linalg.norm(plain.mean(0) - CENTERS[0])
d_neg_to_neg   = np.linalg.norm(neg.mean(0) - CENTERS[0])
print(f'普通CFG 均值 {np.round(plain.mean(0),2)}  到负类别0距离 {d_plain_to_neg:.2f}')
print(f'负引导  均值 {np.round(neg.mean(0),2)}  到负类别0距离 {d_neg_to_neg:.2f}')
assert d_neg_to_neg > d_plain_to_neg, '负引导应把样本更强推离负类别'
print('✅ 负引导：把基线换成负类别，样本被主动推离「不想要的内容」')

---
## ✏️ 练习 1：CFG 公式与边界

实现 `cfg(eps_c, eps_uncond, w)`（约定 `ε̃=ε_∅+w(ε_c−ε_∅)`），并实现 `equivalent_form(eps_c, eps_uncond, w)` 用**等价的另一种写法** `(1+s)·ε_c − s·ε_∅`（其中 `s=w−1`）——验证两种写法在所有 w 下数值相同。

In [ ]:
def cfg(eps_c, eps_uncond, w):
    # TODO: ε_∅ + w·(ε_c − ε_∅)
    raise NotImplementedError

def equivalent_form(eps_c, eps_uncond, w):
    # TODO: 用 s=w-1，返回 (1+s)·ε_c − s·ε_∅，应与 cfg 等价
    raise NotImplementedError

In [ ]:
# —— 练习 1 自测 ——
ec = rng.standard_normal((50, 2)); eu = rng.standard_normal((50, 2))
assert np.allclose(cfg(ec, eu, 0.0), eu), 'w=0→ε_∅'
assert np.allclose(cfg(ec, eu, 1.0), ec), 'w=1→ε_c'
# 两种写法等价
for w in [0.0, 1.0, 2.5, 7.5]:
    assert np.allclose(cfg(ec, eu, w), equivalent_form(ec, eu, w)), f'两写法应等价 @w={w}'
print('✅ 练习 1 通过：CFG 公式正确，两种等价写法一致')

## ✏️ 练习 2：强度权衡度量

实现 `on_target_fraction(samples)`：返回落在**类别1 一侧**（`x>0`）的样本比例。

用它验证：朝类别1 采样时，`w=6` 的保真度严格高于 `w=0`（用提供的 `sample_cfg`）。

In [ ]:
def on_target_fraction(samples):
    # TODO: 返回 (samples[:,0] > 0) 的比例
    raise NotImplementedError

In [ ]:
# —— 练习 2 自测 ——
s0 = sample_cfg(2000, target_c=1, w=0.0, seed=11)
s6 = sample_cfg(2000, target_c=1, w=6.0, seed=11)
f0, f6 = on_target_fraction(s0), on_target_fraction(s6)
print(f'w=0 保真 {f0:.2f}   w=6 保真 {f6:.2f}')
assert 0.0 <= f0 <= 1.0 and 0.0 <= f6 <= 1.0
assert f6 > f0 + 0.2, '更大的 w 应显著提升保真度'
assert f6 > 0.9, 'w=6 应几乎全在目标侧'
print('✅ 练习 2 通过：量化了「w 越大越扣条件」')

## ✏️ 练习 3：负引导

实现 `cfg_negative(eps_target, eps_neg, w)`：把基线换成负预测 `ε_neg`，返回 `ε_neg + w·(ε_target − ε_neg)`。

验证：① `w=1` 时退化为 `ε_target`（与基线无关）；② 引导向量 `ε̃−ε_neg` 与 `(ε_target−ε_neg)` 同向。

In [ ]:
def cfg_negative(eps_target, eps_neg, w):
    # TODO: ε_neg + w·(ε_target − ε_neg)
    raise NotImplementedError

In [ ]:
# —— 练习 3 自测 ——
et = rng.standard_normal((40, 2)); en = rng.standard_normal((40, 2))
# w=1 退化为目标（基线被完全抵消）
assert np.allclose(cfg_negative(et, en, 1.0), et), 'w=1 应退化为 ε_target'
# w=0 退化为负基线
assert np.allclose(cfg_negative(et, en, 0.0), en), 'w=0 应等于 ε_neg'
# 引导方向同向
out = cfg_negative(et, en, 5.0)
guide = out - en; want = et - en
cos = (guide * want).sum() / (np.linalg.norm(guide) * np.linalg.norm(want) + 1e-12)
assert cos > 0.99, '引导应沿 ε_neg→ε_target 方向'
print('✅ 练习 3 通过：负引导外推正确（w=1 退化、方向正确）')

## ✏️ 练习 4：动态 / 调度 CFG

实现 `cosine_guidance_schedule(t, T, w_max)`：让引导强度随去噪步**变化**——这里用「早期强、后期弱」的余弦衰减：
`w(t) = w_max · 0.5·(1 + cos(π·(T-1-t)/(T-1)))`，使 `t=T-1`（最早、最噪）时 `w≈w_max`，`t=0`（最后、最干净）时 `w≈0`。

In [ ]:
def cosine_guidance_schedule(t, T, w_max):
    # TODO: 返回 w_max * 0.5*(1+cos(pi*(T-1-t)/(T-1)))
    #       t=T-1 -> ≈w_max（早期强）; t=0 -> ≈0（后期弱）
    raise NotImplementedError

In [ ]:
# —— 练习 4 自测 ——
T = 200; wmax = 8.0
w_early = cosine_guidance_schedule(T - 1, T, wmax)   # 最早一步
w_late  = cosine_guidance_schedule(0, T, wmax)       # 最后一步
w_mid   = cosine_guidance_schedule((T - 1) // 2, T, wmax)
print(f'w(早期t={T-1})={w_early:.2f}  w(中段)={w_mid:.2f}  w(后期t=0)={w_late:.2f}')
assert abs(w_early - wmax) < 0.1, '最早一步应≈w_max'
assert abs(w_late) < 0.1, '最后一步应≈0'
assert w_early > w_mid > w_late, '应单调：早期强、后期弱'
print('✅ 练习 4 通过：动态 CFG —— 早期强引导定结构、后期弱引导保自然')

---
### 📖 参考答案（先自己做，再对照）

In [ ]:
# 练习 1 参考答案
def cfg(eps_c, eps_uncond, w):
    return eps_uncond + w * (eps_c - eps_uncond)
def equivalent_form(eps_c, eps_uncond, w):
    s = w - 1.0
    return (1 + s) * eps_c - s * eps_uncond

In [ ]:
# 练习 2 参考答案
def on_target_fraction(samples):
    return float((samples[:, 0] > 0).mean())

In [ ]:
# 练习 3 参考答案
def cfg_negative(eps_target, eps_neg, w):
    return eps_neg + w * (eps_target - eps_neg)

In [ ]:
# 练习 4 参考答案
def cosine_guidance_schedule(t, T, w_max):
    return w_max * 0.5 * (1 + np.cos(np.pi * (T - 1 - t) / (T - 1)))

---
## 🧪 真实数据胶囊：真实模型的引导强度

不同真实文生图模型推荐的 **CFG 引导强度**差别很大——因为最优 w 依赖模型、训练、采样器。下面是公开的常用默认值。

算一笔实用账：CFG 要跑两次网络（条件+无条件），相对不用引导，**采样算力翻倍**。这正是少步模型要把 CFG 蒸馏掉的动机。

In [ ]:
# 真实模型常用 CFG 引导强度（公开默认/推荐值，约数）
CFG_DEFAULTS = {
    'SD 1.5':       7.5,
    'SDXL':         7.0,
    'SD3':          7.0,
    'SDXL-Turbo':   0.0,    # 蒸馏后无需 CFG（单步）
    'Flux schnell': 0.0,    # 引导蒸馏进模型
}
print(f"{'模型':<14}{'默认CFG w':>10}{'需两次前向?':>14}")
for name, w in CFG_DEFAULTS.items():
    needs_two = 'YES (2x算力)' if w > 1.0 else 'NO (已蒸馏)'
    print(f'{name:<14}{w:>10.1f}{needs_two:>16}')
print('\n观察：经典 SD 用 w≈7、需两次前向；Turbo/schnell 把 CFG 蒸馏掉 -> w=0、单次前向')

**🧪 胶囊练习**：实现 `sampling_cost_multiplier(w)`：若 `w>1` 需条件+无条件两次前向（返回 2.0），否则单次（返回 1.0）。

并实现 `nfe_with_cfg(steps, w)`：给定采样步数与 w，返回总的网络前向次数（NFE）。

In [ ]:
def sampling_cost_multiplier(w):
    # TODO: w>1 -> 2.0 (要算 ε_c 和 ε_∅)，否则 1.0
    raise NotImplementedError

def nfe_with_cfg(steps, w):
    # TODO: steps * sampling_cost_multiplier(w)
    raise NotImplementedError

In [ ]:
# 自测
assert sampling_cost_multiplier(7.5) == 2.0, '开 CFG 要两次前向'
assert sampling_cost_multiplier(0.0) == 1.0, '不开 CFG 单次前向'
assert sampling_cost_multiplier(1.0) == 1.0, 'w=1 等于普通条件，单次'
# 50 步 SD 开 CFG = 100 次前向；4 步 Turbo 不开 = 4 次
assert nfe_with_cfg(50, 7.5) == 100, '50步×2 = 100 NFE'
assert nfe_with_cfg(4, 0.0) == 4, '4步Turbo单次 = 4 NFE'
print('50步 SD 开 CFG = 100 NFE；4步 Turbo 免 CFG = 4 NFE（25倍差距）')
print('✅ 胶囊练习通过：理解 CFG 的算力代价与蒸馏掉它的动机')

In [ ]:
# 📖 胶囊参考答案
def sampling_cost_multiplier(w):
    return 2.0 if w > 1.0 else 1.0
def nfe_with_cfg(steps, w):
    return steps * sampling_cost_multiplier(w)

---
### 小结
- **CFG = 一个网络两种预测 + 外推**：训练时随机丢条件让网络同时会 `ε_c` 和 `ε_∅`，采样时 `ε̃=ε_∅+w(ε_c−ε_∅)`。
- **边界**：w=0→无条件、w=1→普通条件、w>1→沿条件方向外推放大（无需任何额外分类器）。
- **等价于分类器**：`ε_c−ε_∅ ∝ ∇log p(c|x)`，CFG 从 `p(x)·p(c|x)^w` 采样——锐化条件、**保真↑多样↓**。
- **w 是核心旋钮**：常用 7~12，但**不是越大越好**——过大→过饱和、伪影（外推出数据流形）。缓解：CFG rescale、动态/区间引导。
- **负引导**：把基线 `ε_∅` 换成不想要内容 `ε_neg`，同时吸引目标 + 排斥负例。
- **代价**：要两次前向、算力翻倍——这正是少步模型（Turbo/schnell）把 CFG 蒸馏掉的动机（接模块 05）。

下一站：**模块 05 · 一致性模型与少步采样** —— 把上百步的采样压成一步，让实时生成成为可能。